# MediFlow - Parte 1
## IA Multimodal, Agentes & Lógica de Decisión

## INSTALACIONES

In [1]:
# instalaciones

!pip install -q google-genai pydantic

In [56]:
## importaciones y constantes

from google import genai
from google.colab import userdata

API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=API_KEY)

print("Cliente Gemini configurado correctamente.")

Cliente Gemini configurado correctamente.


In [70]:
# función para reintentar conexión a modelo

import time

def consultar_gemini(prompt, intentos=3):
    for intento in range(intentos):
        try:
            respuesta = client.models.generate_content(
                model="gemini-3.1-flash-lite",
                contents=prompt
            )
            return respuesta.text

        except Exception as e:
            print(f"Intento {intento + 1}/{intentos} falló: {e}")

            if intento < intentos - 1:
                espera = 2 ** intento
                print(f"Reintentando en {espera} segundos...")
                time.sleep(espera)
            else:
                raise

## DATOS DE ENTRADA

In [4]:
# documento clínico de prueba

documento_clinico = """
HOSPITAL SANTA LUCÍA

INFORME DE ESTUDIO RADIOLOGICO

Paciente: Carlos Eduardo Mendes
Edad: 52 años

Médico solicitante: Dra. Renata Silveira
Matrícula: 145892

Estudio: Tomografía de Tórax con contraste

Indicación:
Sospecha de embolia pulmonar aguda. Disnea súbita.

Hallazgos:
Defecto de llenado en arteria pulmonar principal derecha
compatible con TEP agudo.

Conclusión:
Cuadro compatible con Tromboembolismo Pulmonar Agudo.
Se sugiere correlación clínica urgente.

Canal de origen: Guardia de Emergencias
"""

In [5]:
# prompt para extracción

prompt_extraccion = f"""
Eres un sistema de procesamiento documental clínico.

Analiza el siguiente documento y extrae la información clínica y administrativa relevante.

Documento:

{documento_clinico}

Identifica:

- tipo de document
- paciente
- edad
- médico solicitante
- matrícula
- estudio realizado
- diagnóstico o conclusión
- indicación clínica
- nivel de urgencia
- canal de origen

Devuelve la información de forma clara y estructurada
"""

resultado = consultar_gemini(prompt_extraccion)

print(resultado)

Aquí tienes la información extraída y estructurada a partir del documento clínico procesado:

---

### **PROCESAMIENTO DE DOCUMENTO CLÍNICO**

* **Tipo de documento:** Informe de estudio radiológico
* **Paciente:** Carlos Eduardo Mendes
* **Edad:** 52 años
* **Médico solicitante:** Dra. Renata Silveira
* **Matrícula:** 145892
* **Estudio realizado:** Tomografía de Tórax con contraste
* **Indicación clínica:** Sospecha de embolia pulmonar aguda / Disnea súbita
* **Diagnóstico / Conclusión:** Cuadro compatible con Tromboembolismo Pulmonar Agudo (TEP agudo) en la arteria pulmonar principal derecha
* **Nivel de urgencia:** Alta / Urgente (se sugiere correlación clínica urgente)
* **Canal de origen:** Guardia de Emergencias


## MODELADO DE DATOS

In [83]:
# deficiones de pydantic / modelos de datos

from enum import Enum
from typing import Optional
from pydantic import BaseModel

class Paciente(BaseModel):
    nombre: str
    edad: int

class MedicoSolicitante(BaseModel):
    nombre: Optional[str] = None
    matricula: Optional[str] = None

class NivelPrioridad(str, Enum):
    RUTINA = "Rutina"
    URGENTE = "Urgente"
    AMBIGUO = "Ambiguo"

class ClinicalDocument(BaseModel):
    tipo_documento: str
    canal_origen: str
    paciente: Paciente
    medico_solicitante: MedicoSolicitante
    estudio: str
    indicacion_clinica: str
    diagnostico: str
    nivel_prioridad: NivelPrioridad

class EvaluacionConsistencia(BaseModel):
    hay_inconsistencias: bool
    conflictos: list[str]

In [73]:
# documento de prueba

documento = ClinicalDocument(
    tipo_documento="Informe de Estudio por Imágenes",
    canal_origen="Guardia de Emergencias",
    paciente=Paciente(
        nombre="Carlos Eduardo Mendes",
        edad=52
    ),
    medico_solicitante=MedicoSolicitante(
        nombre="Dra. Renata Silveira",
        matricula="145892"
    ),
    estudio="Tomografía de Tórax con contraste",
    indicacion_clinica="Sospecha de embolia pulmonar aguda, disnea súbita",
    diagnostico="Tromboembolismo Pulmonar Agudo (TEP)",
    nivel_prioridad="Urgente"
)

print(documento)

tipo_documento='Informe de Estudio por Imágenes' canal_origen='Guardia de Emergencias' paciente=Paciente(nombre='Carlos Eduardo Mendes', edad=52) medico_solicitante=MedicoSolicitante(nombre='Dra. Renata Silveira', matricula='145892') estudio='Tomografía de Tórax con contraste' indicacion_clinica='Sospecha de embolia pulmonar aguda, disnea súbita' diagnostico='Tromboembolismo Pulmonar Agudo (TEP)' nivel_prioridad=<NivelPrioridad.URGENTE: 'Urgente'>


## EXTRACCIÓN ESTRUCTURADA CON LLM

In [80]:
# prompt extracción json

prompt_json = f"""
Eres un sistema de extracción de información clínica.

Analiza el siguiente documento y extrae exclusivamente la información solicitada.

DOCUMENTO {documento_clinico}

Devuelve únicamente un objeto JSON válido, sin Markdown y sin explicaciones adicionales.

La estructura debe ser exactamente:

{{
   "tipo_documento": "...",
   "canal_origen": "...",
    "paciente": {{
        "nombre": "...",
        "edad": 0
    }},
    "medico_solicitante": {{
        "nombre": "...",
        "matricula": "..."
    }},
    "estudio": "...",
    "indicacion_clinica": "...",
    "diagnostico": "...",
    "nivel_prioridad": "Rutina | Urgente | Ambiguo"
}}
"""

In [42]:
# ejecución extracción

resultado_json = consultar_gemini(prompt_json)
print(resultado_json)

{
   "tipo_documento": "Informe de estudio radiológico",
   "canal_origen": "Guardia de Emergencias",
    "paciente": {
        "nombre": "Carlos Eduardo Mendes",
        "edad": 52
    },
    "medico_solicitante": {
        "nombre": "Dra. Renata Silveira",
        "matricula": "145892"
    },
    "estudio": "Tomografía de Tórax con contraste",
    "indicacion_clinica": "Sospecha de embolia pulmonar aguda. Disnea súbita.",
    "diagnostico": "Cuadro compatible con Tromboembolismo Pulmonar Agudo.",
    "nivel_prioridad": "Urgente"
}


In [43]:
# parseo json extracción

import json

datos = json.loads(resultado_json)

print(type(datos))
print(datos)

<class 'dict'>
{'tipo_documento': 'Informe de estudio radiológico', 'canal_origen': 'Guardia de Emergencias', 'paciente': {'nombre': 'Carlos Eduardo Mendes', 'edad': 52}, 'medico_solicitante': {'nombre': 'Dra. Renata Silveira', 'matricula': '145892'}, 'estudio': 'Tomografía de Tórax con contraste', 'indicacion_clinica': 'Sospecha de embolia pulmonar aguda. Disnea súbita.', 'diagnostico': 'Cuadro compatible con Tromboembolismo Pulmonar Agudo.', 'nivel_prioridad': 'Urgente'}


In [44]:
# validación ClinicalDocument con pydantic

documento = ClinicalDocument(**datos)
print(documento)

tipo_documento='Informe de estudio radiológico' canal_origen='Guardia de Emergencias' paciente=Paciente(nombre='Carlos Eduardo Mendes', edad=52) medico_solicitante=MedicoSolicitante(nombre='Dra. Renata Silveira', matricula='145892') estudio='Tomografía de Tórax con contraste' indicacion_clinica='Sospecha de embolia pulmonar aguda. Disnea súbita.' diagnostico='Cuadro compatible con Tromboembolismo Pulmonar Agudo.' nivel_prioridad=<NivelPrioridad.URGENTE: 'Urgente'>


## SCORE DE CONFIANZA

#### COMPLETITUD

In [45]:
# calculo del score de completitud

def score_completitud(documento):
    campos = [
        documento.tipo_documento,
        documento.canal_origen,
        documento.paciente.nombre,
        documento.paciente.edad,
        documento.medico_solicitante.nombre,
        documento.medico_solicitante.matricula,
        documento.estudio,
        documento.indicacion_clinica,
        documento.diagnostico,
        documento.nivel_prioridad
    ]

    presentes = sum(
        1 for campo in campos
        if campo is not None and campo != ""
    )

    total = len(campos)

    return presentes / total

In [46]:
# ejemplo calculo score completitud

score = score_completitud(documento)

print("Score de completitud:", score)

Score de completitud: 1.0


#### AMBIGÜEDAD

In [47]:
# modelo de evaluación de ambigüedad / pydantic

from pydantic import BaseModel

class EvaluacionAmbiguedad(BaseModel):
  hay_ambiguedad: bool
  observaciones: list[str]

In [48]:
# prompt ambigüedad

prompt_ambiguedad = f"""
Eres un sistema de evaluación de documentos clínicos.

Tu tarea es determinar si existe ambigüedad en la información extraída del documento.

NO debes determinar si un diagnóstico médico es verdadero o falso.

Debes evaluar únicamente la claridad y confiabilidad de la información extraída.

Busca especialmente:

- información ambigua o poco clara
- contradicciones entre distintas partes del documento
- información insuficiente para interpretar correctamente un campo
- datos que podrían tener más de una interpretación
- diagnósticos expresados como sospecha o posibilidad
- información incompleta que pueda afectar la clasificación o el triaje

DOCUMENTO ORIGINAL:

{documento_clinico}

DATOS EXTRAÍDOS:

{documento.model_dump_json()}

Devuelve unicamente un objeto JSON válido.

La estructura debe ser exactamente:

{{
    "hay_ambiguedad": true,
    "observaciones": [
        "Descripción de la ambigüedad encontrada"
    ]
}}

Si no existe ninguna ambigüedad relevante, devuelve:

{{
    "hay_ambiguedad": false,
    "observaciones": []
}}

No inventes información que no aparezca en el documento.
"""

In [53]:
# función para limpieza de respuesta json

def limpiar_json(respuesta):
    respuesta = respuesta.strip()

    if respuesta.startswith("```json"):
        respuesta = respuesta[7:]

    if respuesta.endswith("```"):
        respuesta = respuesta[:-3]

    return respuesta.strip()

In [58]:
# ejemplo score ambigüedad sin limpiar

resultado_ambiguedad = consultar_gemini(prompt_ambiguedad)

print(resultado_ambiguedad)

Intento 1/3 falló: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Reintentando en 1 segundos...
Intento 2/3 falló: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Reintentando en 2 segundos...
{
    "hay_ambiguedad": false,
    "observaciones": []
}


In [59]:
# ejemplo score ambigüedad con parseo + limpiar (quitar markdown)

import json

respuesta_limpia = limpiar_json(resultado_ambiguedad)

datos_ambiguedad = json.loads(respuesta_limpia)

evaluacion_ambiguedad = EvaluacionAmbiguedad(**datos_ambiguedad)

print("¿Hay ambigüedad?:", evaluacion_ambiguedad.hay_ambiguedad)
print("Observaciones:", evaluacion_ambiguedad.observaciones)

¿Hay ambigüedad?: False
Observaciones: []


In [60]:
# ejemplo documento ambiguo

documento_ambiguo = """
HOSPITAL SANTA LUCÍA

INFORME DE ESTUDIO

Paciente: Juan Pérez
Edad: 47 años

Estudio: Tomografía de tórax

Indicación:
Disnea y dolor torácico.

Hallazgos:
Se observa una imagen compatible con posible defecto de
llenado en una rama pulmonar. El hallazgo debe correlacionarse
con el cuadro clínico.

Conclusión:
Posible tromboembolismo pulmonar. No se puede confirmar
el diagnóstico con los hallazgos actuales.

Canal de origen: Consulta externa
"""

In [84]:
# prompt específico para ejemplo documento ambiguo

prompt_json_ambiguo = f"""
Eres un sistema de extracción de información clínica.

Analiza el siguiente documento y extrae exclusivamente
la información solicitada.

DOCUMENTO:
{documento_ambiguo}

Devuelve únicamente un objeto JSON válido, sin explicaciones.

La estructura debe ser exactamente:

{{
    "tipo_documento": "...",
    "canal_origen": "...",
    "paciente": {{
        "nombre": "...",
        "edad": 0
    }},
    "medico_solicitante": {{
        "nombre": "...",
        "matricula": "..."
    }},
    "estudio": "...",
    "indicacion_clinica": "...",
    "diagnostico": "...",
    "nivel_prioridad": "Rutina | Urgente | Ambiguo"
}}

Si un dato no aparece en el documento, no lo inventes.
"""

In [85]:
# ejemplo: ejecución consulta extracción + parseo + limpiar 'documento ambiguo'

resultado_json_ambiguo = consultar_gemini(prompt_json_ambiguo)

respuesta_limpia_ambigua = limpiar_json(resultado_json_ambiguo)

datos_ambiguos = json.loads(respuesta_limpia_ambigua)

documento_ambiguo_validado = ClinicalDocument(**datos_ambiguos)

print(documento_ambiguo_validado)

Intento 1/5 falló: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Reintentando en 1 segundos...
tipo_documento='Informe de estudio' canal_origen='Consulta externa' paciente=Paciente(nombre='Juan Pérez', edad=47) medico_solicitante=MedicoSolicitante(nombre=None, matricula=None) estudio='Tomografía de tórax' indicacion_clinica='Disnea y dolor torácico' diagnostico='Posible tromboembolismo pulmonar' nivel_prioridad=<NivelPrioridad.AMBIGUO: 'Ambiguo'>


In [86]:
# ejemplo: score completitud 'documento ambiguo'

score_completitud_ambiguo = score_completitud(documento_ambiguo_validado)

print("Score de completitud:", score_completitud_ambiguo)

Score de completitud: 0.8


#### CONSISTENCIA

In [92]:
# prompt para consistencia

prompt_consistencia = f"""
Eres un sistema de evaluación de documentos clínicos.

Tu tarea es evaluar la CONSISTENCIA entre:

1. El documento clínico original.
2. Los datos que fueron extraídos del documento.

Busca únicamente posibles contradicciones o conflictos entre
la información del documento y los datos extraídos.

Considera, entre otros:

- datos del paciente
- edad
- médico solicitante
- estudio
- indicación clínica
- diagnóstico
- nivel de prioridad
- cualquier otro dato relevante que aparezca en el documento

No debes determinar si un diagnóstico médico es verdadero o falso.

No debes inventar información que no aparezca en el documento.

DOCUMENTO ORIGINAL:

{documento_ambiguo}

DATOS EXTRAÍDOS:

{documento_ambiguo_validado.model_dump_json()}

Devuelve únicamente un objeto JSON válido.

La estructura debe ser exactamente:

{{
    "hay_inconsistencias": false,
    "conflictos": []
}}

Si encuentras una o más inconsistencias:

{{
    "hay_inconsistencias": true,
    "conflictos": [
        "Descripción concreta del conflicto encontrado"
    ]
}}
"""

In [91]:
# ejemplo: ejecucion + limpieza json + parseo (evaluacion 'documento ambiguo')

resultado_consistencia = consultar_gemini(prompt_consistencia)

respuesta_limpia = limpiar_json(resultado_consistencia)

datos_consistencia = json.loads(respuesta_limpia)

evaluacion_consistencia = EvaluacionConsistencia(**datos_consistencia)

print("¿Hay inconsistencias?:", evaluacion_consistencia.hay_inconsistencias)
print("Conflictos:", evaluacion_consistencia.conflictos)

Intento 1/5 falló: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Reintentando en 1 segundos...
Intento 2/5 falló: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Reintentando en 2 segundos...
Intento 3/5 falló: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Reintentando en 4 segundos...
¿Hay inconsistencias?: False
Conflictos: []


#### CALCULO SCORE CONFIANZA

In [93]:
# función para cálculo de confianza (completitud, ambiguedad, consisencia)

def calcular_confianza(completitud, hay_inconsistencias, hay_ambiguedad):
    consistencia = 0.0 if hay_inconsistencias else 1.0
    claridad = 0.0 if hay_ambiguedad else 1.0

    confianza = (
        completitud * 0.50
        + consistencia * 0.30
        + claridad * 0.20
    )

    return confianza

In [94]:
# ejemplo calculo socre confianza (en 'documento ambiguo')

confianza = calcular_confianza(
    score_completitud_ambiguo,
    evaluacion_consistencia.hay_inconsistencias,
    evaluacion_ambiguedad.hay_ambiguedad
)

print(f"Confianza final: {confianza:.0%}")

Confianza final: 90%


## GRAFO DE DECISIÓN

In [95]:
# función 'nodo decisión' para decidir ruta

def decidir_ruta(confianza, hay_inconsistencias, hay_ambiguedad, nivel_prioridad):
    if hay_inconsistencias or hay_ambiguedad:
        return "revision_humana"

    if nivel_prioridad == "Urgente":
        return "prioridad_urgente"

    if nivel_prioridad == "Ambiguo":
        return "revision_humana"

    if confianza >= 0.80:
        return "procesamiento_automatico"

    return "revision_humana"

In [96]:
# ejemplo: decidir ruta 'documento ambiguo'

ruta = decidir_ruta(
    confianza,
    evaluacion_consistencia.hay_inconsistencias,
    evaluacion_ambiguedad.hay_ambiguedad,
    documento_ambiguo_validado.nivel_prioridad.value
)

print("Ruta:", ruta)

Ruta: revision_humana


#### LangGraph

In [97]:
!pip install -q langgraph

In [98]:
from typing import TypedDict

class EstadoMediFlow(TypedDict):
    confianza: float
    hay_inconsistencias: bool
    hay_ambiguedad: bool
    nivel_prioridad: str
    ruta: str
    mensaje: str

In [99]:
def nodo_decision(estado: EstadoMediFlow):
    ruta = decidir_ruta(
        estado["confianza"],
        estado["hay_inconsistencias"],
        estado["hay_ambiguedad"],
        estado["nivel_prioridad"]
    )

    return {"ruta": ruta}

In [100]:
def determinar_ruta(estado: EstadoMediFlow):
    return estado["ruta"]

In [101]:
def procesamiento_automatico(estado):
    return {
        "ruta": "procesamiento_automatico",
        "mensaje": "Documento listo para procesamiento automático."
    }

def revision_humana(estado):
    return {
        "ruta": "revision_humana",
        "mensaje": "Documento enviado a revisión humana."
    }

def prioridad_urgente(estado):
    return {
        "ruta": "prioridad_urgente",
        "mensaje": "Documento marcado como prioritario."
    }

In [102]:
from langgraph.graph import StateGraph, START, END

grafo = StateGraph(EstadoMediFlow)

grafo.add_node("decision", nodo_decision)
grafo.add_node("procesamiento_automatico", procesamiento_automatico)
grafo.add_node("revision_humana", revision_humana)
grafo.add_node("prioridad_urgente", prioridad_urgente)

grafo.add_edge(START, "decision")

grafo.add_conditional_edges(
    "decision",
    determinar_ruta,
    {
        "procesamiento_automatico": "procesamiento_automatico",
        "revision_humana": "revision_humana",
        "prioridad_urgente": "prioridad_urgente"
    }
)

grafo.add_edge("procesamiento_automatico", END)
grafo.add_edge("revision_humana", END)
grafo.add_edge("prioridad_urgente", END)

grafo = grafo.compile()

In [163]:
estado_inicial = {
    "confianza": confianza,
    "hay_inconsistencias": evaluacion_consistencia.hay_inconsistencias,
    "hay_ambiguedad": evaluacion_ambiguedad.hay_ambiguedad,
    "nivel_prioridad": documento_ambiguo_validado.nivel_prioridad.value,
    "ruta": "",
    "mensaje": ""
}

In [164]:
resultado = grafo.invoke(estado_inicial)

print("Ruta final:", resultado["ruta"])
print("Acción:", resultado["mensaje"])

Ruta final: revision_humana
Acción: Documento enviado a revisión humana.
